In [3]:
!pip install langchain langchain-community langchain-huggingface sentence-transformers chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.1 MB/s eta 0:00:00


#Embedding và Test RETRIEVER

In [2]:
import json
import torch
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN VÀ THIẾT BỊ
# ==========================================
INPUT_FILE = "/content/chunks_perfect_final_group_A (2).json"
DB_DIR = "/content/chroma_db_group_A"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Hệ thống đang chạy trên thiết bị: {device.upper()}")

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH EMBEDDING (BẢN FIX LỖI LANGCHAIN)
# ==========================================
print("⏳ Đang tải mô hình BKAI Bi-Encoder...")
embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True},
    show_progress=True
)
print("✅ Đã tải xong mô hình Embedding!")

# ==========================================
# 3. ĐỌC DỮ LIỆU & CHUYỂN ĐỔI (CÓ THANH TIẾN TRÌNH)
# ==========================================
print(f"\n⏳ Đang đọc dữ liệu từ {INPUT_FILE}...")
docs = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

# Bọc chunks_data vào tqdm để xem tiến độ lặp
for item in tqdm(chunks_data, desc="Đang đóng gói Documents", unit="chunk"):
    metadata = {
        "doc_id": item.get("doc_id", "unknown"),
        "chunk_index": item.get("chunk_index", 0),
        "zone": item.get("zone", "body"),
        "breadcrumbs": item.get("breadcrumbs", "")
    }
    doc = Document(page_content=item["text"], metadata=metadata)
    docs.append(doc)

print(f"✅ Đã đóng gói xong {len(docs)} Documents. Bắt đầu đưa vào GPU để Vector hóa...")

# ==========================================
# 4. NHÚNG VÀ LƯU VÀO CHROMADB
# ==========================================
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=DB_DIR
)
print(f" HOÀN TẤT! Vector Database đã được lưu an toàn tại: {DB_DIR}")

# ==========================================
# 5. TEST THỬ TÌM KIẾM (RETRIEVER)
# ==========================================
print("\n ĐANG TEST TÌM KIẾM NGỮ NGHĨA...")
query = "Nhà Mạc quản lý lãnh thổ và phân chia đạo Thừa tuyên như thế nào?"

# Lấy ra 3 đoạn chunk liên quan nhất
retrieved_docs = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- TOP {i} MATCH ---")
    print(f" Nguồn: {doc.metadata['breadcrumbs']}")
    # In ra 200 ký tự đầu tiên để kiểm tra
    print(f"Trích đoạn: {doc.page_content[:200]}...")

🖥️ Hệ thống đang chạy trên thiết bị: CUDA
⏳ Đang tải mô hình BKAI Bi-Encoder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

🎉 HOÀN TẤT! Vector Database đã được lưu an toàn tại: /content/chroma_db_group_A

🔍 ĐANG TEST TÌM KIẾM NGỮ NGHĨA...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- 🌟 TOP 1 MATCH ---
📌 Nguồn: Lãnh thổ nước Đại Việt thời Mạc
📝 Trích đoạn: --- TÀI LIỆU: NHÀ MIẠC VỚI SỰ NGHIỆP BẢ0 VỆ QUYỀN LỢI QUỐC 81A ---
--- VỊ TRÍ: Lãnh thổ nước Đại Việt thời Mạc ---

Trong khi, nhà Lê (Nam triều) chỉ lấy
lại được vùng đất nhỏ hơn (4 đạo), gồm 17
phủ,...

--- 🌟 TOP 2 MATCH ---
📌 Nguồn: Nhà Mạc thực thi chủ quyền trên
📝 Trích đoạn: --- TÀI LIỆU: NHÀ MIẠC VỚI SỰ NGHIỆP BẢ0 VỆ QUYỀN LỢI QUỐC 81A ---
--- VỊ TRÍ: Nhà Mạc thực thi chủ quyền trên ---

vùng lãnh thổ thuộc quyền quản lý

Sau khi tiếp quản lãnh thổ của đất nước
do nhà Lê...

--- 🌟 TOP 3 MATCH ---
📌 Nguồn: Nhà Mạc thực thi chủ quyền trên
📝 Trích đoạn: --- TÀI LIỆU: NHÀ MIẠC VỚI SỰ NGHIỆP BẢ0 VỆ QUYỀN LỢI QUỐC 81A ---
--- VỊ TRÍ: Nhà Mạc thực thi chủ quyền trên ---

Nhà Mạc tăng cường quản lý ruộng
đất uà cư dân tại các làng xã

Không chỉ đưa quan l...


#Generation

In [9]:
!pip install -U bitsandbytes accelerate transformers

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("⏳ Đang kết nối lại với Vector Database trên ổ cứng...")

embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

DB_DIR = "/content/chroma_db_group_A"
vectorstore = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)

print("✅ Đã kết nối thành công với vectorstore!")

⏳ Đang kết nối lại với Vector Database trên ổ cứng...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_8622/92005990.py:15: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)


✅ Đã kết nối thành công với vectorstore!


In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 5.1. KHỞI TẠO LOCAL LLM (Qwen2.5-3B-Instruct)
# ==========================================
print("\n⏳ Đang tải mô hình Qwen2.5-3B-Instruct...")

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
)

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.1,
    top_p=0.9,
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
print("✅ Đã tải xong LLM Local (Qwen 3B)!")

# ==========================================
# 5.2. THIẾT LẬP RAG CHAIN (TỐI ƯU CHO TÓM TẮT & KHOA HỌC)
# ==========================================
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

prompt_template = """<|im_start|>system
Bạn là một chuyên gia phân tích và tóm tắt tài liệu. Nhiệm vụ của bạn là đọc các đoạn "Ngữ cảnh" được trích xuất từ tài liệu và thực hiện yêu cầu của người dùng (tóm tắt toàn cục hoặc trả lời câu hỏi).
Quy tắc tối thượng:
1. Chỉ dựa vào thông tin có trong phần Ngữ cảnh.
2. Trình bày khoa học, rõ ràng (dùng gạch đầu dòng nếu cần).
3. Tuyệt đối không bịa đặt thêm thông tin. Nếu ngữ cảnh không có thông tin, hãy nói rõ: "Tài liệu hiện tại không cung cấp thông tin về vấn đề này."<|im_end|>
<|im_start|>user
Ngữ cảnh trích xuất từ tài liệu:
{context}

Yêu cầu/Câu hỏi của tôi: {question}<|im_end|>
<|im_start|>assistant
"""

prompt = PromptTemplate.from_template(prompt_template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ==========================================
# 5.3. CHẠY THỬ NGHIỆM TÓM TẮT & HỎI ĐÁP
# ==========================================
print("\n🤖 HỆ THỐNG RAG VỚI QWEN SẴN SÀNG! Đang xử lý yêu cầu...")

query = "Nhà Mạc quản lý lãnh thổ và phân chia đạo Thừa tuyên như thế nào?"

response = rag_chain.invoke(query)

print("\n" + "="*60)
print("🗣️ YÊU CẦU CỦA BẠN:", query)
print("-" * 60)
print(" QWEN TRẢ LỜI:")
print(response.strip())
print("="*60)


⏳ Đang tải mô hình Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Đã tải xong LLM Local (Qwen 3B)!

🤖 HỆ THỐNG RAG VỚI QWEN SẴN SÀNG! Đang xử lý yêu cầu...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🗣️ YÊU CẦU CỦA BẠN: Nhà Mạc quản lý lãnh thổ và phân chia đạo Thừa tuyên như thế nào?
------------------------------------------------------------
🤖 QWEN TRẢ LỜI:
Theo ngữ cảnh, Nhà Mạc quản lý lãnh thổ và phân chia đạo Thừa tuyên như sau:

1. Nhà Mạc quản lý lãnh thổ do nhà Lê sơ để lại và sau khi bị chia tách một phần cho nhà Lê.

2. Nhà Mạc quản lý lãnh thổ bằng cách đưa quan lại, quân lính đến trấn giữ và cai quản đất đai tại các đạo Thừa tuyên.

3. Nhà Mạc xác lập chủ quyền của mình trên toàn bộ lãnh thổ của đất nước bằng cách đưa quan, quân đến trấn giữ những vùng đất xa, như Thanh-Nghệ, Thuận-Quảng và các vùng đất lân cận thuộc các đạo ở phía Bắc.

4. Nhà Mạc thiết lập hệ thống chính quyền các cấp từ trung ương xuống địa phương, bao gồm đạo, phủ, huyện, châu, xã và thôn.

5. Nhà Mạc cai quản các đạo Thừa tuyên thông qua việc phân chia các đạo thành các ty: Thừa ty, Đô ty và Hiến ty.

6. Nhà Mạc đặt 3 ty ở các đạo như thời Lê Thánh Tông, bao gồm Giám sát Ngự sử, Tham chính ở đạo

#Lưu và cách gọi mô hình

In [11]:
from google.colab import drive
import shutil
import os

# Bước 1: Kết nối với Google Drive
drive.mount('/content/drive')

# Bước 2: Định nghĩa đường dẫn
source_path = '/content/chroma_db_group_A'
zip_filename = 'chroma_db_group_A'
drive_path = '/content/drive/MyDrive/RAG_Project/' # Bạn có thể thay đổi tên thư mục trên Drive

# Tạo thư mục trên Drive nếu chưa có
if not os.path.exists(drive_path):
    os.makedirs(drive_path)

# Bước 3: Nén thư mục thành file .zip
print(f"⏳ Đang nén thư mục {source_path}...")
shutil.make_archive(zip_filename, 'zip', source_path)

# Bước 4: Di chuyển file .zip vào Google Drive
print(f"🚚 Đang chuyển file vào Drive tại: {drive_path}")
shutil.move(f"{zip_filename}.zip", os.path.join(drive_path, f"{zip_filename}.zip"))

print("✅ Hoàn tất! Database của bạn đã được lưu an toàn trên Drive.")

Mounted at /content/drive
⏳ Đang nén thư mục /content/chroma_db_group_A...
🚚 Đang chuyển file vào Drive tại: /content/drive/MyDrive/RAG_Project/
✅ Hoàn tất! Database của bạn đã được lưu an toàn trên Drive.


In [12]:
from google.colab import drive
import shutil
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Kết nối Drive và giải nén
drive.mount('/content/drive')
zip_path = '/content/drive/MyDrive/RAG_Project/chroma_db_group_A.zip'
extract_path = '/content/chroma_db_group_A'

if os.path.exists(zip_path):
    print("⏳ Đang giải nén Database từ Drive...")
    shutil.unpack_archive(zip_path, extract_path, 'zip')
    print("✅ Giải nén thành công.")

# 2. Khởi tạo lại Embedding (phải dùng cùng loại với lúc tạo DB)
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': device}
)

# 3. Gọi lại Vector Store
vectorstore = Chroma(persist_directory=extract_path, embedding_function=embeddings)
print("🚀 Database đã sẵn sàng để truy vấn!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Đang giải nén Database từ Drive...
✅ Giải nén thành công.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🚀 Database đã sẵn sàng để truy vấn!
